# Baseline v2: LoRA vs AdaLoRA vs L1RA на Qwen3-0.6B

Повторный прогон для анализа **ранней диагностики важности модулей**. Отличия от `baseline.ipynb`:

- Только одна модель: `Qwen/Qwen3-0.6B`
- Чекпоинты адаптера сохраняются **каждые 50 optimizer-шагов** (в `baseline.ipynb` сохранялся только финал — `_CHECKPOINT_STEPS` был мёртвым кодом)
- Мгновенный loss логируется **на каждом optimizer-шаге** (`log_per_step=True`), а не раз в 100 шагов
- Вывод идёт в `./logs_v2/`, чтобы не перезаписать существующие `./runs/`

Требует обновлённого `src/trainer.py` с параметрами `save_steps` и `log_per_step`.

## 0. Colab: mount Drive (локально пропустить)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#!pip install -U packaging ninja
#!pip install -U flash-attn --no-build-isolation

In [ ]:
%cd /content/drive/MyDrive/diploma/project

## 1. Imports & config

In [ ]:
import sys
sys.path.insert(0, ".")

import gc
import json
import os
import random

import numpy as np
import torch
import pandas as pd
from functools import partial
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from src import (
    SFTDataset,
    collate_fn,
    filter_df_by_max_len,
    build_lora_model,
    build_adalora_model,
    build_l1ra_model,
    build_optimizer,
    build_l1ra_optimizer,
    build_scheduler,
    train_model,
)
from src.warm_start import WarmStartDetector


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # cuDNN: deterministic but slower. Acceptable trade-off for reproducibility.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

In [ ]:
SEED         = 42
MODEL_NAME   = "Qwen/Qwen3-0.6B"
MAX_LENGTH   = 1024
BATCH_SIZE   = 8
NUM_EPOCHS   = 2
GRAD_ACCUM   = 4
LR           = 1e-4
LORA_RANKS   = [16]   # param-matched с target_r=16 у AdaLoRA
TRAIN_SIZE   = 40_000
VAL_SIZE     = 4_000
DATA_DIR     = "./data"
RUNS_DIR     = "./logs_v2"   # старые 20k/4ep-прогоны лежат в logs_v2_20k4ep

# фильтр датасетов (open_code_instruct пропускаем — не входит в протокол защиты)
DATASETS_SELECTED = ['meta_math', 'open_orca']

# v2-specific: save every 50 optimizer steps, log loss every step
SAVE_STEPS   = 50
LOG_PER_STEP = True

os.makedirs(RUNS_DIR, exist_ok=True)
print(f"Output dir: {os.path.abspath(RUNS_DIR)}")

## 2. Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## 3. Data preparation

Parquet-файлы должны уже существовать (создаются один раз через `baseline.ipynb`, ячейка со скачиванием). Здесь просто читаем и унифицируем колонки.

In [ ]:
# Скачиваем 50k-выборки с HF, если их ещё нет. Сырые имена колонок сохраняем —
# нормализацию делает следующая ячейка.
N_RAW = 50_000
SEED  = 42

_HF_SPECS = [
    # (fname_prefix, hf_repo, split, keep_raw_cols)
    ('meta-math_MetaMathQA',   'meta-math/MetaMathQA',     'train', ['query', 'response']),
    ('Open-Orca_OpenOrca',     'Open-Orca/OpenOrca',       'train', ['question', 'response']),
    ('nvidia_OpenCodeInstruct','nvidia/OpenCodeInstruct',  'train', ['input', 'output']),
]

_need_download = any(
    not os.path.exists(f'{DATA_DIR}/{prefix}_{N_RAW}.parquet')
    for prefix, *_ in _HF_SPECS
)

if _need_download:
    from datasets import load_dataset
    os.makedirs(DATA_DIR, exist_ok=True)
    for prefix, repo, split, keep in _HF_SPECS:
        out_path = f'{DATA_DIR}/{prefix}_{N_RAW}.parquet'
        if os.path.exists(out_path):
            print(f'skip (exists): {out_path}')
            continue
        print(f'downloading {repo} ...')
        ds = load_dataset(repo, split=split)
        ds = ds.shuffle(seed=SEED).select(range(N_RAW))
        df = ds.to_pandas()[keep]
        df.to_parquet(out_path, index=False)
        print(f'  -> {out_path}  shape={df.shape}  cols={list(df.columns)}')
else:
    print(f'All 50k parquets already present in {DATA_DIR}')


In [ ]:
# N_RAW задан в ячейке с download-ом (50_000).
meta_math = (
    pd.read_parquet(f"{DATA_DIR}/meta-math_MetaMathQA_{N_RAW}.parquet")
    .rename(columns={"query": "question", "response": "answer"})
    [["question", "answer"]]
)
open_code = (
    pd.read_parquet(f"{DATA_DIR}/nvidia_OpenCodeInstruct_{N_RAW}.parquet")
    .rename(columns={"input": "question", "output": "answer"})
    [["question", "answer"]]
)
open_orca = (
    pd.read_parquet(f"{DATA_DIR}/Open-Orca_OpenOrca_{N_RAW}.parquet")
    [["question", "response"]]
    .rename(columns={"response": "answer"})
)
print(f'raw rows: meta_math={len(meta_math)}  open_code={len(open_code)}  open_orca={len(open_orca)}')


In [ ]:
meta_math_filtered, _ = filter_df_by_max_len(meta_math, tokenizer, MAX_LENGTH)
open_code_filtered, _ = filter_df_by_max_len(open_code, tokenizer, MAX_LENGTH)
open_orca_filtered, _ = filter_df_by_max_len(open_orca, tokenizer, MAX_LENGTH)

In [ ]:
def split_train_val(df, train_size=TRAIN_SIZE, val_size=VAL_SIZE, name=''):
    need = train_size + val_size
    assert len(df) >= need, (
        f'[{name}] после фильтрации по max_len осталось {len(df)} строк, нужно {need}. '
        f'Подними N_RAW или уменьши TRAIN/VAL.'
    )
    train = df.iloc[:train_size].reset_index(drop=True)
    val   = df.iloc[train_size:train_size + val_size].reset_index(drop=True)
    return train, val

_filtered = {
    'meta_math':          meta_math_filtered,
    'open_code_instruct': open_code_filtered,
    'open_orca':          open_orca_filtered,
}

_splits = {}
for _name in DATASETS_SELECTED:
    _tr, _va = split_train_val(_filtered[_name], name=_name)
    _splits[_name] = (_tr, _va)
    print(f'{_name:20s}  train: {len(_tr)}  val: {len(_va)}')


## 4. DataLoaders

In [ ]:
def make_loaders(train_df, val_df, seed: int = SEED):
    train_ds = SFTDataset(train_df, tokenizer, max_length=MAX_LENGTH)
    val_ds   = SFTDataset(val_df,   tokenizer, max_length=MAX_LENGTH)
    _collate = partial(collate_fn, tokenizer=tokenizer)
    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=_collate, generator=g,
    )
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=_collate)
    return train_loader, val_loader

# DATASETS уже отфильтрован по DATASETS_SELECTED в предыдущей ячейке.
DATASETS = _splits
print('Будут прогнаны датасеты:', list(DATASETS.keys()))

## 5. LoRA (r=16)


In [ ]:
LR = 1e-4
lora_runs = []

for ds_name, (train_df, val_df) in DATASETS.items():
    seed_everything(SEED)
    train_loader, val_loader = make_loaders(train_df, val_df, seed=SEED)

    for rank in LORA_RANKS:
        run_name   = f"{ds_name}_lora_r{rank}"
        output_dir = f"{RUNS_DIR}/{run_name}"
        print(f"\n>>> {run_name}  (seed={SEED})")

        model     = build_lora_model(rank, model_name=MODEL_NAME)
        optimizer = build_optimizer(model, lr=LR)
        scheduler = build_scheduler(optimizer, len(train_loader), NUM_EPOCHS, GRAD_ACCUM)

        detector = WarmStartDetector(
            save_steps=SAVE_STEPS, window=100, k_consecutive=3, threshold=0.90,
        )

        logs = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            output_dir=output_dir,
            num_epochs=NUM_EPOCHS,
            grad_accum_steps=GRAD_ACCUM,
            device="cuda",
            save_steps=SAVE_STEPS,
            log_per_step=LOG_PER_STEP,
            detector=detector,
        )
        model.save_pretrained(f"{output_dir}/adapter")
        lora_runs.append({
            "dataset": ds_name, "rank": rank, "output_dir": output_dir, "logs": logs,
            "warm_start_step": detector.trigger_step,
        })

        del model
        torch.cuda.empty_cache(); gc.collect()

## 6. AdaLoRA (init_r=32, target_r=16)


In [ ]:
ADALORA_CONFIGS = [
    {"init_r": 32, "target_r": 16},
]

LR = 1e-3
adalora_runs = []

for ds_name, (train_df, val_df) in DATASETS.items():
    seed_everything(SEED)
    train_loader, val_loader = make_loaders(train_df, val_df, seed=SEED)
    total_step = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS

    for cfg in ADALORA_CONFIGS:
        run_name   = f"{ds_name}_adalora_init{cfg['init_r']}_target{cfg['target_r']}"
        output_dir = f"{RUNS_DIR}/{run_name}"
        print(f"\n>>> {run_name}  (total_step={total_step}, seed={SEED})")

        model     = build_adalora_model(**cfg, model_name=MODEL_NAME, total_step=total_step)
        optimizer = build_optimizer(model, lr=LR)
        scheduler = build_scheduler(optimizer, len(train_loader), NUM_EPOCHS, GRAD_ACCUM)

        detector = WarmStartDetector(
            save_steps=SAVE_STEPS, window=100, k_consecutive=3, threshold=0.90,
        )

        logs = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            output_dir=output_dir,
            num_epochs=NUM_EPOCHS,
            grad_accum_steps=GRAD_ACCUM,
            device="cuda",
            save_steps=SAVE_STEPS,
            log_per_step=LOG_PER_STEP,
            detector=detector,
        )
        model.save_pretrained(f"{output_dir}/adapter")
        adalora_runs.append({
            "dataset": ds_name, **cfg, "output_dir": output_dir, "logs": logs,
            "warm_start_step": detector.trigger_step,
        })

        del model
        torch.cuda.empty_cache(); gc.collect()

## 7. L1RA (r=32, λ=1e-3)


In [ ]:
L1RA_CONFIGS = [
    {"rank": 32, "l1ra_lambda": 1e-3},
]

LR = 1e-3
l1ra_runs = []

for ds_name, (train_df, val_df) in DATASETS.items():
    seed_everything(SEED)
    train_loader, val_loader = make_loaders(train_df, val_df, seed=SEED)

    for cfg in L1RA_CONFIGS:
        lam        = cfg["l1ra_lambda"]
        run_name   = f"{ds_name}_l1ra_r{cfg['rank']}_lam{lam}"
        output_dir = f"{RUNS_DIR}/{run_name}"
        print(f"\n>>> {run_name}  (seed={SEED})")

        model     = build_l1ra_model(rank=cfg["rank"], model_name=MODEL_NAME, l1ra_lambda=lam)
        optimizer = build_l1ra_optimizer(model, lr=LR)
        scheduler = build_scheduler(optimizer, len(train_loader), NUM_EPOCHS, GRAD_ACCUM)

        detector = WarmStartDetector(
            save_steps=SAVE_STEPS, window=100, k_consecutive=3, threshold=0.90,
        )

        logs = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            output_dir=output_dir,
            num_epochs=NUM_EPOCHS,
            grad_accum_steps=GRAD_ACCUM,
            device="cuda",
            save_steps=SAVE_STEPS,
            log_per_step=LOG_PER_STEP,
            detector=detector,
        )

        adapter_dir = f"{output_dir}/adapter"
        os.makedirs(adapter_dir, exist_ok=True)
        torch.save(
            {n: p for n, p in model.named_parameters() if p.requires_grad},
            f"{adapter_dir}/adapter_model.pt",
        )
        with open(f"{adapter_dir}/adapter_config.json", "w") as f:
            json.dump(model.peft_config["default"].__dict__, f, indent=2, default=str)

        l1ra_runs.append({
            "dataset": ds_name, **cfg, "output_dir": output_dir, "logs": logs,
            "warm_start_step": detector.trigger_step,
        })

        del model
        torch.cuda.empty_cache(); gc.collect()

## 8. Results summary

In [ ]:
def best_val_ppl(logs):
    val_rows = [r for r in logs if r.get("split") == "val" and r.get("ppl")]
    return min(r["ppl"] for r in val_rows) if val_rows else None

rows = []
for run in lora_runs:
    rows.append({"method": "LoRA", "dataset": run["dataset"],
                 "config": f"r={run['rank']}",
                 "best_val_ppl": best_val_ppl(run["logs"])})
for run in adalora_runs:
    rows.append({"method": "AdaLoRA", "dataset": run["dataset"],
                 "config": f"init{run['init_r']}\u2192{run['target_r']}",
                 "best_val_ppl": best_val_ppl(run["logs"])})
for run in l1ra_runs:
    rows.append({"method": "L1RA", "dataset": run["dataset"],
                 "config": f"r={run['rank']},\u03bb={run['l1ra_lambda']}",
                 "best_val_ppl": best_val_ppl(run["logs"])})

summary = pd.DataFrame(rows).sort_values(["dataset", "best_val_ppl"]).reset_index(drop=True)
summary.to_csv(f"{RUNS_DIR}/summary.csv", index=False)
summary

## 9. Sanity check: что реально сохранилось

После прогона должны быть чекпоинты `step_50, step_100, ..., step_~626` в каждом `logs_v2/*/checkpoints/`.

In [ ]:
import glob

for run_dir in sorted(glob.glob(f"{RUNS_DIR}/*/")):
    ckpts = sorted(glob.glob(f"{run_dir}/checkpoints/step_*"))
    metrics = f"{run_dir}/metrics.csv"
    n_log = 0
    if os.path.exists(metrics):
        n_log = len(pd.read_csv(metrics))
    print(f"{os.path.basename(os.path.normpath(run_dir)):50s}  checkpoints={len(ckpts):3d}  metric_rows={n_log}")